In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from lightgbm import LGBMClassifier

In [ ]:
# Chargement des données
df = pd.read_csv("../data/datos_arbolado_clean.csv")

df = df.dropna(subset=["estado_plantera"])

cat_cols = df.select_dtypes(include=["object"]).columns

encoders = {col: LabelEncoder().fit(df[col]) for col in cat_cols}
for col, enc in encoders.items():
    df[col] = enc.transform(df[col])

X = df.drop(columns=["estado_plantera"])
y = df["estado_plantera"]

KeyError: ['estado_planta']

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
# GridSearch LightGBM
param_grid = {
    "num_leaves": [31, 50],
    "max_depth": [-1, 5, 10],
    "learning_rate": [0.1, 0.01],
    "n_estimators": [200, 500],
    "class_weight": ["balanced"]
}

lgbm = LGBMClassifier(random_state=42)

grid = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best params:", grid.best_params_)

In [ ]:
# Evaluation
y_pred = best_model.predict(X_test)

print("\n=== Classification report ===")
print(classification_report(y_test, y_pred))

print("\n=== Matrice de confusion ===")
print(confusion_matrix(y_test, y_pred))